# Main Program 

In [1]:
import streamlit as st
import os
import time
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_classic.chains import create_retrieval_chain  # 改從 langchain.chains 直接引入
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from langchain_core.documents import Document
from uuid import uuid4

/Users/shenglienlee/anaconda3/envs/Me_RAG/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the .env file
load_dotenv()

# Retrieve the Pinecone API key from user data
pinecone_key = os.environ.get('PINECONE_API_KEY')

# Initialize the OpenAI client with the API key from user data
OpenAI_key=os.environ.get("OPENAI_API_KEY")

# 參數設定
INDEX_NAME = "william-resume-v1"
EMBEDDING_MODEL = "text-embedding-3-large"
DIMENSION = 3072

def load_and_chunk_pdfs(file_paths):
    documents = [] 
    for path in file_paths:
        single_file_text = ""
        try:
            reader = PdfReader(path)
            #extract text from each page of pdf and combine them together
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    single_file_text  = single_file_text + text + " "
        except Exception as e:
            print(f"   ⚠️ 無法讀取 {path}: {e}")
            continue
            
        # 情況 A: 處理「履歷表」 (使用 Regex 做結構化切分)
        if "履歷" in path:
            print(f"   Using [Regex Strategy] for Resume...")
            # I chunk my resume by the section title since each section are not related.
            resume_separators = [
                r"(學歷|工作經驗|專業技能|李盛廉)", # Priority 1: 抓大章節
                "\n \n"
            ]
            text_splitter = RecursiveCharacterTextSplitter(
                separators=resume_separators,
                is_separator_regex=True, # 關鍵：啟用 Regex 支援
                keep_separator=True,     # 關鍵：保留標題 (如 "學歷") 在 Chunk 開頭
                chunk_size=1000,         # 設定大一點，確保整個 Section 完整
                chunk_overlap=0          # Section 之間通常不需要重疊
            )
            
            single_file_chunks = text_splitter.split_text(single_file_text)
            # 過濾過短的雜訊
            single_file_chunks = [c.strip() for c in single_file_chunks if len(c.strip()) > 10]
            doc_type = "resume"

        # 情況 B: 處理「自傳」 (標準遞迴切分)
        else:
            print(f"   Using [Standard Strategy] for Autobiography...")
            
            # 自傳不需要 Regex，使用預設的 separators 即可
            # 這裡設定 is_separator_regex=False (預設值)，separators 用一般的字串列表
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=100,
                separators=["\n \n", "\n", "。", "！", "？", " ", ""],
                is_separator_regex=False 
            )
            single_file_chunks = text_splitter.split_text(single_file_text)
            doc_type = "biography"
        
        # ----------------
        
        # --- 3. 包裝成 Document 物件 (關鍵修改) ---
        
        for chunk in single_file_chunks:
            # 建立 Document 物件
            doc = Document(
                page_content=chunk,  # 👈 內文保持乾淨 (Clean Text)
                metadata={           # 👈 來源資訊藏在這裡
                    "source": path,
                    "type": doc_type
                }
            )
            documents.append(doc)

        print(f"   -> {path} 建立了 {len(single_file_chunks)} 個 Document 物件。")

    print(f"✅ 所有文件處理完畢，共 {len(documents)} 個物件。")
    return documents

def init_vector_store(documents):
    """2. 初始化 Pinecone 並上傳向量 (Upsert)"""
    # 初始化 Embedding 模型
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    # 初始化 Pinecone
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

    # 檢查並建立 Index
    existing_indexes = [i.name for i in pc.list_indexes()]
    if INDEX_NAME not in existing_indexes:
        print(f"🚀 建立 Pinecone Index: {INDEX_NAME} (Serverless)...")
        pc.create_index(
            name=INDEX_NAME,
            dimension=DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
        
        # 等待初始化
        while not pc.describe_index(INDEX_NAME).status.ready:
            time.sleep(1)
    index = pc.Index(INDEX_NAME)

    print("📤 正在將向量上傳至 Pinecone (這可能需要幾秒鐘)...")
    vector_store = PineconeVectorStore(index=index, embedding=embeddings)
    #create unique id for each chunk
    uuids = [str(uuid4()) for _ in range(len(documents))]
    vector_store.add_documents(documents=documents, ids=uuids)

    print("✅ 向量資料庫準備完成！")

    return vector_store

def setup_rag_chain(vector_store):
    """3. 建立檢索生成鏈 (Chain)"""
    
    # 設定 LLM (GPT-4o)
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)

    # 設定檢索器 (Retriever)
    pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
    index = pc.Index(INDEX_NAME)
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vector_store = PineconeVectorStore(index=index, embedding=embeddings)
    retriever = vector_store.as_retriever(
        search_type="similarity", 
        search_kwargs={"k": 3} # 取前3個最相關的片段
    )

    # 設定 Prompt
    #{context} 佔位符： 這是一個變數。等一下程式執行時，
    # Retriever 抓到的那 3 個履歷片段，會自動被塞進這個 {context} 的位置。
    system_prompt = (
        "You are an AI assistant representing Sheng-Lien Lee (William) for an interview at ASUS.\n"
        "Answer strictly based on the context provided.\n"
        "Context: {context}"
    )
    #{input} 佔位符： 這是使用者的問題。
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    # 組合 Chain
    #create_stuff_documents_chain:這條鏈只負責：「把資料塞進 Prompt -> 叫 LLM 回答」。
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)
    #這是更高一層的邏輯，它把「找資料的人 (retriever)」和「處理資料的人 (combine_docs_chain)」串起來。
    rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

    return rag_chain

# --- 主程式執行區 ---
if __name__ == "__main__":
    # 指定檔案路徑
    path_1 = r"/Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉_履歷.pdf"
    path_2 = r"/Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉＿自傳.pdf"
    pdf_files = [path_1,path_2]
    
    # 1. 處理資料
    documents = load_and_chunk_pdfs(pdf_files)
    
    # 2. 準備向量庫
    vector_store = init_vector_store(documents)
    
    # 3. 建立 RAG 鏈
    rag_chain = setup_rag_chain(vector_store)
    
    # 4. 互動測試迴圈
    print("\n" + "="*30)
    print("🤖 RAG Bot 已就緒 (輸入 'exit' 離開)")
    print("="*30)
    
    while True:
        query = input("\n請輸入問題: ")
        if query.lower() in ["exit", "quit"]:
            break
            
        print("Thinking...")
        response = rag_chain.invoke({"input": query})
        
        print(f"\n💡 回答:\n{response['answer']}")
        
        # 顯示引用來源 (Debug用)
        # print("\n🔍 參考片段 (Context):")
        # for i, doc in enumerate(response["context"]):
        #     print(f"--- Chunk {i+1} ---")
        #     print(doc.page_content[:100] + "...") # 只印前100字

   Using [Regex Strategy] for Resume...
   -> /Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉_履歷.pdf 建立了 8 個 Document 物件。
   Using [Standard Strategy] for Autobiography...
   -> /Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉＿自傳.pdf 建立了 3 個 Document 物件。
✅ 所有文件處理完畢，共 11 個物件。
📤 正在將向量上傳至 Pinecone (這可能需要幾秒鐘)...
✅ 向量資料庫準備完成！

🤖 RAG Bot 已就緒 (輸入 'exit' 離開)
Thinking...

💡 回答:
I am Sheng-Lien Lee, a data scientist with a business background, holding a Master's degree in Data Science from the University of Rochester in the United States. I have over two years of practical experience in data engineering and AI applications. I am skilled in independently designing and deploying data pipelines and AI models. I have participated in various interdisciplinary, cross-departmental, and cross-cultural projects, focusing on transforming data insights into actionable strategies and 

# Test

In [2]:
load_dotenv()

# Retrieve the Pinecone API key from user data
pinecone_key = os.environ.get('PINECONE_API_KEY')

# Initialize the OpenAI client with the API key from user data
OpenAI_key=os.environ.get("OPENAI_API_KEY")

# 參數設定
INDEX_NAME = "william-resume-v1"
EMBEDDING_MODEL = "text-embedding-3-large"
DIMENSION = 3072

## Chunk PDF

In [9]:

"""1. 載入 PDF 並切分文字"""
print("📄 正在讀取 PDF...")
path_1 = r"/Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉_履歷.pdf"
path_2 = r"/Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉＿自傳.pdf"
file_paths = [path_1,path_2]
full_text = ""

"""
混合切分策略 (Hybrid Chunking) - Modern Approach:
使用統一的 RecursiveCharacterTextSplitter，但透過 is_separator_regex 參數
來區分「結構化履歷」與「敘事型自傳」的切分邏輯。
"""

documents = [] 
for path in file_paths:
    single_file_text = ""
    try:
        reader = PdfReader(path)
        #extract text from each page of pdf and combine them together
        for page in reader.pages:
            text = page.extract_text()
            if text:
                single_file_text  = single_file_text + text + " "
    except Exception as e:
        print(f"   ⚠️ 無法讀取 {path}: {e}")
        continue
        
    # 情況 A: 處理「履歷表」 (使用 Regex 做結構化切分)
    if "履歷" in path:
        print(f"   Using [Regex Strategy] for Resume...")
        # I chunk my resume by the section title since each section are not related.
        resume_separators = [
            r"(學歷|工作經驗|專業技能|李盛廉)", # Priority 1: 抓大章節
            "\n \n"
        ]
        text_splitter = RecursiveCharacterTextSplitter(
            separators=resume_separators,
            is_separator_regex=True, # 關鍵：啟用 Regex 支援
            keep_separator=True,     # 關鍵：保留標題 (如 "學歷") 在 Chunk 開頭
            chunk_size=1000,         # 設定大一點，確保整個 Section 完整
            chunk_overlap=0          # Section 之間通常不需要重疊
        )
        
        single_file_chunks = text_splitter.split_text(single_file_text)
        # 過濾過短的雜訊
        single_file_chunks = [c.strip() for c in single_file_chunks if len(c.strip()) > 10]
        doc_type = "resume"

    # 情況 B: 處理「自傳」 (標準遞迴切分)
    else:
        print(f"   Using [Standard Strategy] for Autobiography...")
        
        # 自傳不需要 Regex，使用預設的 separators 即可
        # 這裡設定 is_separator_regex=False (預設值)，separators 用一般的字串列表
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            separators=["\n \n", "\n", "。", "！", "？", " ", ""],
            is_separator_regex=False 
        )
        single_file_chunks = text_splitter.split_text(single_file_text)
        doc_type = "biography"
    
    # ----------------
    
    # --- 3. 包裝成 Document 物件 (關鍵修改) ---
    
    for chunk in single_file_chunks:
        # 建立 Document 物件
        doc = Document(
            page_content=chunk,  # 👈 內文保持乾淨 (Clean Text)
            metadata={           # 👈 來源資訊藏在這裡
                "source": path,
                "type": doc_type
            }
        )
        documents.append(doc)

    print(f"   -> {path} 建立了 {len(single_file_chunks)} 個 Document 物件。")

print(f"✅ 所有文件處理完畢，共 {len(documents)} 個物件。")

📄 正在讀取 PDF...
   Using [Regex Strategy] for Resume...
   -> /Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉_履歷.pdf 建立了 8 個 Document 物件。
   Using [Standard Strategy] for Autobiography...
   -> /Users/shenglienlee/Documents/GitHub/quick-start-guide-to-llms?tab=readme-ov-file/notebooks/My_project/doc/李盛廉＿自傳.pdf 建立了 3 個 Document 物件。
✅ 所有文件處理完畢，共 11 個物件。


## Vector DB

### 

In [14]:
"""2. 初始化 Pinecone 並上傳向量 (Upsert)"""
INDEX_NAME = "william-resume-v1"
EMBEDDING_MODEL = "text-embedding-3-large"
DIMENSION = 3072
# 初始化 Embedding 模型
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# 初始化 Pinecone
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))

# 檢查並建立 Index
existing_indexes = [i.name for i in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    print(f"🚀 建立 Pinecone Index: {INDEX_NAME} (Serverless)...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    index = pc.Index(INDEX_NAME)
    # 等待初始化
    while not pc.describe_index(INDEX_NAME).status.ready:
        time.sleep(1)

print("📤 正在將向量上傳至 Pinecone (這可能需要幾秒鐘)...")
vector_store = PineconeVectorStore(index=index, embedding=embeddings)
#create unique id for each chunk
uuids = [str(uuid4()) for _ in range(len(documents))]
vector_store.add_documents(documents=documents, ids=uuids)

print("✅ 向量資料庫準備完成！")


🚀 建立 Pinecone Index: william-resume-v1 (Serverless)...
📤 正在將向量上傳至 Pinecone (這可能需要幾秒鐘)...
✅ 向量資料庫準備完成！


## Set RAG

In [ ]:
"""3. 建立檢索生成鏈 (Chain)"""
    
# 設定 LLM (GPT-4o)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.1)

# 設定檢索器 (Retriever)
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index(INDEX_NAME)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3} # 取前3個最相關的片段
)

# 設定 Prompt
#{context} 佔位符： 這是一個變數。等一下程式執行時，
# Retriever 抓到的那 3 個履歷片段，會自動被塞進這個 {context} 的位置。
system_prompt = (
    "You are an AI assistant representing Sheng-Lien Lee (William) for an interview at ASUS.\n"
    "Answer strictly based on the context provided.\n"
    "Context: {context}"
)
#{input} 佔位符： 這是使用者的問題。
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 組合 Chain
#create_stuff_documents_chain:這條鏈只負責：「把資料塞進 Prompt -> 叫 LLM 回答」。
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
#這是更高一層的邏輯，它把「找資料的人 (retriever)」和「處理資料的人 (combine_docs_chain)」串起來。
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)
